# Final round 04: CatBoost with built-in text features and InSet lexicon counts on the pseudo-labelled reviews

In [18]:
import pandas as pd
import numpy as np
import nltk, string
from nlp_id.stopword import StopWord
from nlp_id.lemmatizer import Lemmatizer
from nlp_id.tokenizer import Tokenizer
from nlp_id.postag import PosTag
from gensim.models.fasttext import load_facebook_model
from tqdm import tqdm
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import seaborn as sns
import matplotlib.pyplot as plt

import tensorflow as tf
import torch
from transformers import AutoTokenizer, TFAutoModelForSequenceClassification, TFBertModel, TrainingArguments, Trainer, \
    AutoModelForSequenceClassification, EarlyStoppingCallback
# import evaluate

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, recall_score, precision_score

In [19]:
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [20]:
df = pd.read_csv('train_iter4.csv')
df.head()

,source,review_id,user_name,review_title,review_description,rating,thumbs_up,review_date,developer_response,developer_response_date,appVersion,laguage_code,country_code,expedition,sentimen
0,Google Play,2c6973de-c75a-405d-9b9e-e5bf233f200a,Bayu Winalas,NaN,Hingga ini masih baik-baik aja 🥰🥰🥰,5,0,2021-09-02 03:36:26,NaN,NaN,3.13.1,id,id,J&T,1.0
1,Google Play,94fde3c7-4a37-4c92-875c-2f33d585f490,Wahyono 345,NaN,"Terima kasih JNE,selama ini selalu pakai jasa ...",5,1,2020-12-14 14:38:40,NaN,NaN,1.1.14,id,id,JNE,1.0
2,Google Play,31927262-6bd2-4ec5-abe5-757dc7df9a7e,Pengguna Google,NaN,ok lah bisa ngebantu,5,0,2019-03-12 15:29:01,NaN,NaN,3.1.1,id,id,J&T,1.0
3,Google Play,9f71ba44-d15c-438b-b3d0-22f861137f03,Pengguna Google,NaN,langganan,5,0,2019-02-28 00:50:52,NaN,NaN,3.1.1,id,id,J&T,1.0
4,Google Play,b409cdd4-1998-4d48-a486-8a5846e446f4,Pengguna Google,NaN,T O P BGT . . Pengiriman super cepat bgt. .suk...,5,0,2016-05-04 12:17:49,NaN,NaN,NaN,id,id,J&T,1.0


In [21]:
stopwords = StopWord().get_stopword()
lemmatizer = Lemmatizer()
tokenizer = Tokenizer()
postagger = PosTag()

c:\Anaconda\envs\py39\lib\site-packages\sklearn\base.py:318: UserWarning: Trying to unpickle estimator DictVectorizer from version 0.22 when using version 1.2.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Anaconda\envs\py39\lib\site-packages\sklearn\base.py:318: UserWarning: Trying to unpickle estimator DecisionTreeClassifier from version 0.22 when using version 1.2.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Anaconda\envs\py39\lib\site-packages\sklearn\base.py:318: UserWarning: Trying to unpickle estimator RandomForestClassifier from version 0.22 when using version 1.2.2. This might lead to breaking code or invalid results. Use at your own

In [22]:
def tokenize_lemmatize(text):
    lemmatized_doc = lemmatizer.lemmatize(text)
    return [word for word in tokenizer.tokenize(lemmatized_doc) if word not in stopwords]

In [23]:
df['review_description_tokenized'] = df['review_description'].apply(tokenize_lemmatize)
df['review_description_tokenized'].head()

0                                             [hingga]
1    [terima, kasih, jne, pakai, jasa, jne, lancar,...
2                                       [ok, ngebantu]
3                                            [langgan]
4    [t, o, p, bgt, kirim, super, cepat, bgt, suka,...
Name: review_description_tokenized, dtype: object

In [24]:
# RQ: Apakah opinionated words mempunyai pengaruh terhadap label_pemerintah?


# Fajri Koto, and Gemala Y. Rahmaningtyas "InSet Lexicon: Evaluation of a Word List for
# Indonesian Sentiment Analysis in Microblogs". IEEE in the 21st International Conference
# on Asian Language Processing (IALP), Singapore, December 2017.
#
# The InSet Lexicon consists of 3,609 positive words and 6,609 negative words with weight ranging from -5 to +5

!wget https://raw.githubusercontent.com/fajri91/InSet/master/negative.tsv
!wget https://raw.githubusercontent.com/fajri91/InSet/master/positive.tsv

'wget' is not recognized as an internal or external command,
operable program or batch file.


'wget' is not recognized as an internal or external command,
operable program or batch file.


In [25]:
# kita muat kata-kata bersentiment tersebut ke memori
# dalam bentuk python's dictionary
# key = word
# value = sentiment score

def load_sentword(filename):
    file = open(filename, "r")
    file.readline()  # skip header
    sentword_score = {}
    for line in file:
        word, score = line.strip().split("\t")
        score = int(score)
        sentword_score[word] = score
    file.close()
    return sentword_score


sentword_score_neg = load_sentword("negative.tsv")
sentword_score_pos = load_sentword("positive.tsv")


# num positives or negatives
def num(text, sentword_score):
    count = 0
    for word in text:
        if word in sentword_score:
            count += 1
    return count


# num JJ positives or negatives
def num_jj(text, sentword_score):
    count = 0
    for word, pos in postagger.get_pos_tag(" ".join(text)):
        if pos == "JJ":
            if word in sentword_score:
                count += 1
    return count


# print(sentword_score)
print(sentword_score_neg['mencederai'])
print(sentword_score_pos['bagus'])

-4
2


In [26]:
from gensim.models import Word2Vec, FastText

VECTOR_SIZE = 64


In [27]:
df["num_pos_review"] = df.review_description_tokenized. \
    map(lambda x: num(x, sentword_score_pos))

df["num_neg_review"] = df.review_description_tokenized. \
    map(lambda x: num(x, sentword_score_neg))


In [28]:
# ft_model = load_facebook_model('cc.id.300.bin\cc.id.300.bin')

In [29]:
df['review_description_tokenized']

0                                                  [hingga]
1         [terima, kasih, jne, pakai, jasa, jne, lancar,...
2                                            [ok, ngebantu]
3                                                 [langgan]
4         [t, o, p, bgt, kirim, super, cepat, bgt, suka,...
                                ...                        
109390    [5, bintang, j, t, tp, maaf, wkt, cancel, kiri...
109391                      [pra, plnggan, gk, ad, kmplain]
109392    [keren, pesan, jam, 10, pg, sampe, jam, 7, mal...
109393               [moga, pick, up, d, sdkit, paket, hhi]
109394                            [bumiayu, tercover, area]
Name: review_description_tokenized, Length: 109395, dtype: object

In [30]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 109395 entries, 0 to 109394
Data columns (total 18 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   source                        109395 non-null  object 
 1   review_id                     109395 non-null  object 
 2   user_name                     109395 non-null  object 
 3   review_title                  0 non-null       float64
 4   review_description            109395 non-null  object 
 5   rating                        109395 non-null  int64  
 6   thumbs_up                     109395 non-null  int64  
 7   review_date                   109395 non-null  object 
 8   developer_response            1890 non-null    object 
 9   developer_response_date       1890 non-null    object 
 10  appVersion                    78291 non-null   object 
 11  laguage_code                  109395 non-null  object 
 12  country_code                  109395 non-nul

In [31]:
embedded_words = [[ft_model.wv[word] for word in token] for token in tqdm(df['review_description_tokenized'].tolist())]
embedded_words_encoding = [np.mean(embedded_word, axis=0) for embedded_word in tqdm(embedded_words)]
df_embedding = pd.DataFrame(embedded_words_encoding)
df_embedding.head()

NameError: name 'ft_model' is not defined

In [ ]:
embedded_words_encoding

[array([-1.64179541e-02, -4.47055176e-02,  9.12660640e-03,  2.94533074e-02,
         3.11762933e-02,  1.00807790e-02,  7.04308506e-03, -9.43548698e-03,
        -1.46082612e-02, -6.97758794e-02, -5.19829849e-03, -2.83528795e-03,
        -8.51089787e-03, -3.08323260e-02,  5.68026677e-03, -1.89429168e-02,
        -2.12365668e-03,  2.09747627e-03,  2.44892878e-03,  1.74050704e-02,
        -1.48611711e-02, -1.79403924e-05, -4.21423540e-02,  1.44894542e-02,
         2.68465234e-03, -2.48412732e-02, -2.22902931e-02, -5.35344984e-03,
         5.28652146e-02, -3.58521123e-03,  5.02765998e-02, -7.82804657e-03,
         8.40320531e-03,  2.95985229e-02,  2.75431722e-02,  2.20058858e-02,
         1.66990440e-02, -2.27035284e-02,  1.33448038e-02,  2.99576973e-03,
         2.28813058e-03, -2.83247735e-02,  2.72627063e-02,  4.84056491e-03,
        -7.43034715e-03, -2.28181295e-02, -4.04535905e-02,  5.09498715e-02,
        -1.10686142e-02, -1.05126780e-02, -2.78639533e-02,  1.11015607e-02,
        -1.1

In [33]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(df[['review_description','num_pos_review', 'num_neg_review']], df['sentimen'], test_size=0.2, random_state=42)

X_train.shape, X_val.shape, y_train.shape, y_val.shape

((87516, 3), (21879, 3), (87516,), (21879,))

In [37]:
import catboost as cb

cbc_model = cb.CatBoostClassifier(
    text_features=['review_description'],
    eval_metric='F1',  # Change the evaluation metric to F1
    task_type='GPU',
    devices='0',
    early_stopping_rounds=200,
    verbose=100,
    random_seed=42,
    iterations=10000,
    max_depth=8,
    learning_rate=0.03,
)
cbc_model.fit(X_train, y_train, eval_set=(X_val, y_val))


0:	learn: 0.8352885	test: 0.8450704	best: 0.8450704 (0)	total: 55.4ms	remaining: 9m 13s
100:	learn: 0.8538973	test: 0.8610650	best: 0.8610650 (100)	total: 2.97s	remaining: 4m 51s
200:	learn: 0.8603361	test: 0.8663707	best: 0.8664780 (198)	total: 5.51s	remaining: 4m 28s
300:	learn: 0.8642896	test: 0.8707141	best: 0.8709262 (289)	total: 8.17s	remaining: 4m 23s
400:	learn: 0.8673171	test: 0.8718147	best: 0.8721223 (391)	total: 10.7s	remaining: 4m 17s
500:	learn: 0.8700918	test: 0.8729503	best: 0.8731613 (467)	total: 13.4s	remaining: 4m 13s
600:	learn: 0.8729359	test: 0.8735944	best: 0.8738931 (562)	total: 16.3s	remaining: 4m 14s
700:	learn: 0.8748014	test: 0.8744667	best: 0.8746525 (695)	total: 19.2s	remaining: 4m 14s
800:	learn: 0.8767283	test: 0.8761954	best: 0.8761954 (800)	total: 22s	remaining: 4m 12s
900:	learn: 0.8782148	test: 0.8768486	best: 0.8768486 (891)	total: 24.6s	remaining: 4m 8s
1000:	learn: 0.8800726	test: 0.8780173	best: 0.8781306 (991)	total: 27.3s	remaining: 4m 5s
1100:

In [39]:
y_val_pred = cbc_model.predict(X_val)
print(classification_report(y_val, y_val_pred))

              precision    recall  f1-score   support

         0.0       0.92      0.95      0.93     13889
         1.0       0.91      0.85      0.88      7990

    accuracy                           0.91     21879
   macro avg       0.91      0.90      0.91     21879
weighted avg       0.91      0.91      0.91     21879

